In [613]:
import os

os.chdir(r"C:\Users\HP USER\OneDrive\Desktop\transfer-learning-kaduna-house-price")

print("Current folder:", os.getcwd())
print("Files in data folder:", os.listdir("data"))

Current folder: C:\Users\HP USER\OneDrive\Desktop\transfer-learning-kaduna-house-price
Files in data folder: ['data_kaggle.csv', 'Strict_Kaduna_South_Dataset.csv']


In [614]:
import pandas as pd
import numpy as np

Load Malaysia dataset only

In [615]:
import pandas as pd

malaysia_df = pd.read_csv("data/data_kaggle.csv")

print("Original shape:", malaysia_df.shape)
print(malaysia_df.columns.tolist())

Original shape: (53883, 8)
['Location', 'Price', 'Rooms', 'Bathrooms', 'Car Parks', 'Property Type', 'Size', 'Furnishing']


Select only relevant columns

In [616]:
selected_columns = [
    "Rooms",
    "Size",
    "Bathrooms",
    "Furnishing",
    "Property Type",
    "Price"
]

malaysia_df = malaysia_df[selected_columns]

print("After selecting columns:", malaysia_df.shape)
malaysia_df.head()

After selecting columns: (53883, 6)


,Rooms,Size,Bathrooms,Furnishing,Property Type,Price
0,2+1,"Built-up : 1,335 sq. ft.",3.0,Fully Furnished,Serviced Residence,"RM 1,250,000"
1,6,Land area : 6900 sq. ft.,7.0,Partly Furnished,Bungalow,"RM 6,800,000"
2,3,"Built-up : 1,875 sq. ft.",4.0,Partly Furnished,Condominium (Corner),"RM 1,030,000"
3,NaN,NaN,NaN,NaN,NaN,NaN
4,4+1,"Built-up : 1,513 sq. ft.",3.0,Partly Furnished,Condominium (Corner),"RM 900,000"


# Check missing values before cleaning

In [617]:
# Check missing values before cleaning
print("Shape before missing value handling:", malaysia_df.shape)
print(malaysia_df.isnull().sum())

Shape before missing value handling: (53883, 6)
Rooms            1706
Size             1063
Bathrooms        2013
Furnishing       6930
Property Type      25
Price             248
dtype: int64


# Drop rows with missing values, following the base paper approach

In [618]:
# Drop rows with missing values, following the base paper approach
malaysia_df = malaysia_df.dropna()

print("Shape after dropping missing values:", malaysia_df.shape)
print(malaysia_df.isnull().sum())

Shape after dropping missing values: (45207, 6)
Rooms            0
Size             0
Bathrooms        0
Furnishing       0
Property Type    0
Price            0
dtype: int64


In [619]:
# Check current data types before numeric cleaning
malaysia_df.dtypes

Rooms                str
Size                 str
Bathrooms        float64
Furnishing           str
Property Type        str
Price                str
dtype: object

In [620]:
# Clean Rooms column
malaysia_df["Rooms"] = (
    malaysia_df["Rooms"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

# Clean Bathrooms column
malaysia_df["Bathrooms"] = (
    malaysia_df["Bathrooms"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

# Clean Size column
malaysia_df["Size"] = (
    malaysia_df["Size"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

# Clean Price column
malaysia_df["Price"] = (
    malaysia_df["Price"]
    .astype(str)
    .str.replace("RM", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

malaysia_df.head()

,Rooms,Size,Bathrooms,Furnishing,Property Type,Price
0,2.0,1335.0,3.0,Fully Furnished,Serviced Residence,1250000.0
1,6.0,6900.0,7.0,Partly Furnished,Bungalow,6800000.0
2,3.0,1875.0,4.0,Partly Furnished,Condominium (Corner),1030000.0
4,4.0,1513.0,3.0,Partly Furnished,Condominium (Corner),900000.0
5,4.0,7200.0,5.0,Partly Furnished,Bungalow,5350000.0


checking datatype

In [621]:
print(malaysia_df["Size"].dtype)

float64


inspect values

In [622]:
malaysia_df["Size"].head(10)

0     1335.0
1     6900.0
2     1875.0
4     1513.0
5     7200.0
7     3600.0
8       25.0
9      904.0
11      22.0
12    1900.0
Name: Size, dtype: float64

confirm size is numeric again

In [623]:
malaysia_df["Size"].dtype

dtype('float64')

Convert to sqm

In [624]:
malaysia_df["Size"] = malaysia_df["Size"] * 0.092903

Rename for clarity

In [625]:
malaysia_df = malaysia_df.rename(columns={"Size": "Size_sqm"})

In [626]:
malaysia_df["Size_sqm"].head()

0    124.025505
1    641.030700
2    174.193125
4    140.562239
5    668.901600
Name: Size_sqm, dtype: float64

Converting malasia to USD

In [627]:
MYR_TO_USD = 0.25

malaysia_df["Price_USD"] = malaysia_df["Price"] * MYR_TO_USD

malaysia_df[["Price", "Price_USD"]].head()

,Price,Price_USD
0,1250000.0,312500.0
1,6800000.0,1700000.0
2,1030000.0,257500.0
4,900000.0,225000.0
5,5350000.0,1337500.0


Drop price

In [628]:
malaysia_df.drop(columns=["Price"], inplace=True)

## 5. Checking for Failed Numeric Conversion

After converting the selected columns to numerical format, some values may fail to convert properly.  
Such failed values become missing values again, so they are removed before continuing.

In [629]:
# Check missing values after numeric conversion
malaysia_df.isnull().sum()

Rooms            737
Size_sqm          46
Bathrooms          0
Furnishing         0
Property Type      0
Price_USD          0
dtype: int64

In [630]:
# Drop rows where numeric conversion failed
malaysia_df = malaysia_df.dropna()

print("Shape after numeric cleaning:", malaysia_df.shape)
malaysia_df.head()

Shape after numeric cleaning: (44425, 6)


,Rooms,Size_sqm,Bathrooms,Furnishing,Property Type,Price_USD
0,2.0,124.025505,3.0,Fully Furnished,Serviced Residence,312500.0
1,6.0,641.030700,7.0,Partly Furnished,Bungalow,1700000.0
2,3.0,174.193125,4.0,Partly Furnished,Condominium (Corner),257500.0
4,4.0,140.562239,3.0,Partly Furnished,Condominium (Corner),225000.0
5,4.0,668.901600,5.0,Partly Furnished,Bungalow,1337500.0


In [631]:
original_cols = [
    "Rooms",
    "Size_sqm",
    "Bathrooms",
    "Furnishing",
    "Property Type",
    "Price_USD"
]

print("Duplicates before conversion:",
      malaysia_df.duplicated(subset=original_cols).sum())

Duplicates before conversion: 6093


## Removing duplicates based on original Malaysia dataset features

In [632]:
original_cols = [
    "Rooms",
    "Size_sqm",
    "Bathrooms",
    "Furnishing",
    "Property Type",
    "Price_USD"
]

malaysia_df = malaysia_df.drop_duplicates(subset=original_cols)

print("Shape after removing duplicates correctly:", malaysia_df.shape)

Shape after removing duplicates correctly: (38332, 6)


In [633]:
# Remove duplicate rows
malaysia_df = malaysia_df.drop_duplicates()

print("Shape after removing duplicates:", malaysia_df.shape)

Shape after removing duplicates: (38332, 6)


In [634]:
# Remove duplicate rows
malaysia_df = malaysia_df.drop_duplicates()

print("Shape after removing duplicates:", malaysia_df.shape)

Shape after removing duplicates: (38332, 6)


In [635]:
print(malaysia_df.columns)

Index(['Rooms', 'Size_sqm', 'Bathrooms', 'Furnishing', 'Property Type',
       'Price_USD'],
      dtype='str')


In [636]:
malaysia_df.shape

(38332, 6)

Use only one price column

In [637]:
print(malaysia_df.columns)

Index(['Rooms', 'Size_sqm', 'Bathrooms', 'Furnishing', 'Property Type',
       'Price_USD'],
      dtype='str')


Confirm no duplicates remain

In [638]:
original_cols = [
    "Rooms",
    "Size_sqm",
    "Bathrooms",
    "Furnishing",
    "Property Type",
    "Price_USD"
]

print("Remaining duplicates:",
      malaysia_df.duplicated(subset=original_cols).sum())

Remaining duplicates: 0


Confirm no missing values

In [639]:
print(malaysia_df.isnull().sum())

Rooms            0
Size_sqm         0
Bathrooms        0
Furnishing       0
Property Type    0
Price_USD        0
dtype: int64


Confirm correct data types

In [640]:
print(malaysia_df.dtypes)

Rooms            float64
Size_sqm         float64
Bathrooms        float64
Furnishing           str
Property Type        str
Price_USD        float64
dtype: object


Check for unrealistic values

In [641]:
malaysia_df.describe()

,Rooms,Size_sqm,Bathrooms,Price_USD
count,38332.000000,38332.000000,38332.000000,3.833200e+04
mean,3.312089,207.281543,3.171319,4.673070e+05
std,1.331076,873.137867,1.667441,2.132842e+06
min,1.000000,0.000000,1.000000,7.700000e+01
25%,3.000000,85.006245,2.000000,1.500000e+05
50%,3.000000,119.844870,3.000000,2.625000e+05
75%,4.000000,205.036921,4.000000,5.200000e+05
max,20.000000,76180.460000,20.000000,4.000000e+08


Check unique categories (avoid messy categories)

In [642]:
print(malaysia_df["Furnishing"].unique())
print(malaysia_df["Property Type"].unique())

<StringArray>
['Fully Furnished', 'Partly Furnished', 'Unfurnished', 'Unknown']
Length: 4, dtype: str
<StringArray>
[                       'Serviced Residence',
                                  'Bungalow',
                      'Condominium (Corner)',
                       'Semi-detached House',
         '2-sty Terrace/Link House (EndLot)',
                  'Apartment (Intermediate)',
   '2-sty Terrace/Link House (Intermediate)',
                   'Bungalow (Intermediate)',
        'Semi-detached House (Intermediate)',
                         'Bungalow (Corner)',
         'Serviced Residence (Intermediate)',
                               'Condominium',
                'Condominium (Intermediate)',
                      'Condominium (EndLot)',
               'Serviced Residence (Corner)',
   '3-sty Terrace/Link House (Intermediate)',
               'Serviced Residence (Duplex)',
                  '2-sty Terrace/Link House',
         '2-sty Terrace/Link House (Corner)',
 '2.5-sty 

## Removing Unknown Values from Furnishing

In [643]:
malaysia_df = malaysia_df[malaysia_df["Furnishing"] != "Unknown"]

print(malaysia_df["Furnishing"].unique())

<StringArray>
['Fully Furnished', 'Partly Furnished', 'Unfurnished']
Length: 3, dtype: str


In [644]:
malaysia_df['Furnishing'].unique()

<StringArray>
['Fully Furnished', 'Partly Furnished', 'Unfurnished']
Length: 3, dtype: str

feature engineering turning fully furnished to new, partly furnished to fair and unfurnished to old

In [645]:
def map_condition(furnishing):
    f = str(furnishing).strip().lower()
    
    if "fully" in f:
        return "New"
    elif "partly" in f or "partial" in f:
        return "Fair"
    elif "unfurnished" in f:
        return "Old"


In [646]:
malaysia_df["Condition"] = malaysia_df["Furnishing"].apply(map_condition)
malaysia_df[["Furnishing", "Condition"]].head()

,Furnishing,Condition
0,Fully Furnished,New
1,Partly Furnished,Fair
2,Partly Furnished,Fair
4,Partly Furnished,Fair
5,Partly Furnished,Fair


In [647]:
malaysia_df = malaysia_df.drop(columns=["Furnishing"])

In [648]:
print(malaysia_df.columns)

Index(['Rooms', 'Size_sqm', 'Bathrooms', 'Property Type', 'Price_USD',
       'Condition'],
      dtype='str')


## Simplifying Property Type Categories

In [649]:
print(malaysia_df["Property Type"].value_counts())

Property Type
Condominium                            7522
Condominium (Corner)                   4719
Serviced Residence                     4397
Condominium (Intermediate)             4144
Serviced Residence (Intermediate)      2485
                                       ... 
Flat (Penthouse)                          1
2.5-sty Terrace/Link House (Duplex)       1
Apartment (Triplex)                       1
Semi-detached House (SOHO)                1
4.5-sty Terrace/Link House (Corner)       1
Name: count, Length: 93, dtype: int64


In [650]:
print(malaysia_df["Property Type"].unique())

<StringArray>
[                       'Serviced Residence',
                                  'Bungalow',
                      'Condominium (Corner)',
                       'Semi-detached House',
         '2-sty Terrace/Link House (EndLot)',
                  'Apartment (Intermediate)',
   '2-sty Terrace/Link House (Intermediate)',
                   'Bungalow (Intermediate)',
        'Semi-detached House (Intermediate)',
                         'Bungalow (Corner)',
         'Serviced Residence (Intermediate)',
                               'Condominium',
                'Condominium (Intermediate)',
                      'Condominium (EndLot)',
               'Serviced Residence (Corner)',
   '3-sty Terrace/Link House (Intermediate)',
               'Serviced Residence (Duplex)',
                  '2-sty Terrace/Link House',
         '2-sty Terrace/Link House (Corner)',
 '2.5-sty Terrace/Link House (Intermediate)',
         '3-sty Terrace/Link House (Corner)',
         '3-sty Terr

Mapping Function

In [651]:
def map_property_type(value):
    value = str(value).lower().strip()

    # Self-contain: small studio-type units
    if "studio" in value:
        return "Self-contain"

    # Bungalow: detached bungalow or land-type properties
    elif "bungalow" in value or "land" in value:
        return "Bungalow"

    # Duplex: private multi-level / compound-style houses
    elif (
        "terrace" in value
        or "link house" in value
        or "townhouse" in value
        or "semi-detached" in value
        or "duplex" in value
        or "triplex" in value
        or "cluster house" in value
    ):
        return "Duplex"

    # Flat: shared multi-unit buildings
    elif (
        "condominium" in value
        or "apartment" in value
        or "flat" in value
        or "serviced residence" in value
        or "soho" in value
        or "penthouse" in value
    ):
        return "Flat"

    # Safe fallback
    else:
        return "Flat"



### 📌 Apply Mapping


In [652]:
malaysia_df["Property Type"] = malaysia_df["Property Type"].apply(map_property_type)



### 📌 Check Unique Values

Verify result

In [653]:
malaysia_df["Property Type"].unique()

<StringArray>
['Flat', 'Bungalow', 'Duplex', 'Self-contain']
Length: 4, dtype: str



### 📌 Check Distribution

In [654]:
malaysia_df["Property Type"].value_counts()

Property Type
Flat            27448
Duplex           7839
Bungalow         2648
Self-contain       19
Name: count, dtype: int64

In [655]:
malaysia_df.columns

Index(['Rooms', 'Size_sqm', 'Bathrooms', 'Property Type', 'Price_USD',
       'Condition'],
      dtype='str')

load Kaduna dataset

In [656]:
kaduna_df = pd.read_csv("data/Strict_Kaduna_South_Dataset.csv")

print("Kaduna original shape:", kaduna_df.shape)
print(kaduna_df.columns.tolist())

kaduna_df.head()

Kaduna original shape: (37, 11)
['Property_Type', 'Bedrooms', 'Bathrooms', 'Size_sqm', 'Location', 'LGA', 'Condition', 'Security', 'Flood_Risk', 'Proximity', 'Price']


,Property_Type,Bedrooms,Bathrooms,Size_sqm,Location,LGA,Condition,Security,Flood_Risk,Proximity,Price
0,Bungalow,3,2,140,Barnawa,Kaduna South,New,High,Low,Near,48000000
1,Bungalow,3,2,150,Barnawa,Kaduna South,New,High,Low,Near,52000000
2,Bungalow,3,2,135,Barnawa,Kaduna South,Fair,Medium,Low,Near,43000000
3,Bungalow,3,2,145,Barnawa,Kaduna South,New,High,Low,Near,50000000
4,Flat,2,2,100,Kakuri,Kaduna South,Fair,Medium,Medium,Near,25000000



---

# 📌 2. Check Basic Info (Data Types + Null Overview)

In [657]:
kaduna_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Property_Type  37 non-null     str  
 1   Bedrooms       37 non-null     int64
 2   Bathrooms      37 non-null     int64
 3   Size_sqm       37 non-null     int64
 4   Location       37 non-null     str  
 5   LGA            37 non-null     str  
 6   Condition      37 non-null     str  
 7   Security       37 non-null     str  
 8   Flood_Risk     37 non-null     str  
 9   Proximity      37 non-null     str  
 10  Price          37 non-null     int64
dtypes: int64(4), str(7)
memory usage: 3.3 KB



---

# 📌 3. Check Missing Values (Per Column)

In [658]:

kaduna_df.isnull().sum()

Property_Type    0
Bedrooms         0
Bathrooms        0
Size_sqm         0
Location         0
LGA              0
Condition        0
Security         0
Flood_Risk       0
Proximity        0
Price            0
dtype: int64


---

# 📌 4. Check Missing Values (Percentage)

In [659]:
(kaduna_df.isnull().sum() / len(kaduna_df)) * 100

Property_Type    0.0
Bedrooms         0.0
Bathrooms        0.0
Size_sqm         0.0
Location         0.0
LGA              0.0
Condition        0.0
Security         0.0
Flood_Risk       0.0
Proximity        0.0
Price            0.0
dtype: float64



# 📌 5. Inspect Unique Values (Important Before Cleaning)

### 🔹 For Categorical Columns

In [660]:
categorical_cols = kaduna_df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    print(f"\n{col}")
    print(kaduna_df[col].unique())


Property_Type
<StringArray>
['Bungalow', 'Flat', 'Duplex', 'Self-contain']
Length: 4, dtype: str

Location
<StringArray>
[     'Barnawa',       'Kakuri',   'Tudun Wada',  'Kabala West',
     'Kinkinau', 'Kurmin Mashi',   'Television']
Length: 7, dtype: str

LGA
<StringArray>
['Kaduna South', 'kaduna south', 'Kaduna south']
Length: 3, dtype: str

Condition
<StringArray>
['New', 'Fair', 'Old', 'Good', 'Excellent', 'Poor']
Length: 6, dtype: str

Security
<StringArray>
['High', 'Medium', 'Low', 'low']
Length: 4, dtype: str

Flood_Risk
<StringArray>
['Low', 'Medium', 'High', 'low']
Length: 4, dtype: str

Proximity
<StringArray>
['Near', 'Moderate', 'Far']
Length: 3, dtype: str


C:\Users\HP USER\AppData\Local\Temp\ipykernel_24284\690665154.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = kaduna_df.select_dtypes(include=["object"]).columns



---

### 🔹 For Numerical Columns

In [661]:
numerical_cols = kaduna_df.select_dtypes(include=["int64", "float64"]).columns

for col in numerical_cols:
    print(f"\n{col}")
    print("Min:", kaduna_df[col].min())
    print("Max:", kaduna_df[col].max())


Bedrooms
Min: 1
Max: 7

Bathrooms
Min: 1
Max: 7

Size_sqm
Min: 35
Max: 800

Price
Min: 2500000
Max: 350000000


# 📌 6. Check Duplicate Rows

In [662]:
kaduna_df.duplicated().sum()

np.int64(0)

# 📌 7. Inspect Rows With Missing Values

In [663]:
kaduna_df[kaduna_df.isnull().any(axis=1)]

,Property_Type,Bedrooms,Bathrooms,Size_sqm,Location,LGA,Condition,Security,Flood_Risk,Proximity,Price



---

# 📌 8. Check Distribution of Key Features

### Property Type

In [664]:
kaduna_df["Property_Type"].value_counts()

Property_Type
Flat            18
Bungalow        10
Duplex           6
Self-contain     3
Name: count, dtype: int64

In [665]:
### Location
kaduna_df["Location"].value_counts()

Location
Barnawa         13
Kabala West      7
Kakuri           4
Tudun Wada       4
Kinkinau         3
Kurmin Mashi     3
Television       3
Name: count, dtype: int64

In [666]:
### Price
kaduna_df["Price"].describe()

count    3.700000e+01
mean     4.616216e+07
std      6.200885e+07
min      2.500000e+06
25%      1.500000e+07
50%      2.600000e+07
75%      4.500000e+07
max      3.500000e+08
Name: Price, dtype: float64


---

# 🔥 Step 4: Drop Irrelevant Columns (for alignment)

These are NOT in Malaysia:

- LGA ❌  
- Security ❌  
- Flood_Risk ❌  
- Proximity ❌  

In [667]:
kaduna_df = kaduna_df.drop(columns=[
    "LGA", 
    "Security", 
    "Flood_Risk", 
    "Proximity"
])

rename the price

In [668]:
kaduna_df = kaduna_df.rename(columns={"Price": "Price_NGN"})

```python
# Set exchange rate
NGN_TO_USD = 1500

# Convert Price_NGN to USD


In [669]:

# Set exchange rate
NGN_TO_USD = 1500

# Convert Price_NGN to USD
kaduna_df["Price_USD"] = kaduna_df["Price_NGN"] / NGN_TO_USD

Drop Price_NGN column

In [670]:
kaduna_df = kaduna_df.drop(columns=["Price_NGN"])

In [671]:
print(kaduna_df.columns)

Index(['Property_Type', 'Bedrooms', 'Bathrooms', 'Size_sqm', 'Location',
       'Condition', 'Price_USD'],
      dtype='str')


condition unique

In [672]:
kaduna_df["Condition"].unique()

<StringArray>
['New', 'Fair', 'Old', 'Good', 'Excellent', 'Poor']
Length: 6, dtype: str

Apply Mapping to Kaduna 

In [673]:
print("Kaduna current columns:")
print(kaduna_df.columns.tolist())

Kaduna current columns:
['Property_Type', 'Bedrooms', 'Bathrooms', 'Size_sqm', 'Location', 'Condition', 'Price_USD']


In [674]:
print("Malaysia exact columns:")
print(malaysia_df.columns.tolist())

Malaysia exact columns:
['Rooms', 'Size_sqm', 'Bathrooms', 'Property Type', 'Price_USD', 'Condition']


Map Kaduna Condition

In [675]:
# Drop Location safely
# errors='ignore' means if Location is already gone
# it will not throw an error — it will just continue
kaduna_df = kaduna_df.drop(columns=['Location'], 
                            errors='ignore')

print("Kaduna columns:")
print(kaduna_df.columns.tolist())

print("\nKaduna shape:", kaduna_df.shape)

Kaduna columns:
['Property_Type', 'Bedrooms', 'Bathrooms', 'Size_sqm', 'Condition', 'Price_USD']

Kaduna shape: (37, 6)


In [676]:
def map_condition_kaduna(value):
    v = str(value).strip().lower()
    if "new" in v or "good" in v or "excellent" in v:
        return "New"
    elif "fair" in v or "average" in v or "medium" in v:
        return "Fair"
    elif "old" in v or "poor" in v:
        return "Old"
    else:
        return "Fair"

kaduna_df['Condition'] = kaduna_df['Condition'].apply(
    map_condition_kaduna)

print("Kaduna Condition after mapping:")
print(kaduna_df['Condition'].value_counts())

Kaduna Condition after mapping:
Condition
New     19
Fair    11
Old      7
Name: count, dtype: int64


Rename malaysia column

In [677]:
# Rename Malaysia columns to match Kaduna
malaysia_df = malaysia_df.rename(columns={
    'Rooms':         'Bedrooms',
    'Property Type': 'Property_Type'
})

print("Malaysia columns after renaming:")
print(malaysia_df.columns.tolist())

Malaysia columns after renaming:
['Bedrooms', 'Size_sqm', 'Bathrooms', 'Property_Type', 'Price_USD', 'Condition']


feature alingnment for both dataset

In [678]:
SHARED_FEATURES = ['Bedrooms', 'Bathrooms', 'Size_sqm',
                   'Property_Type', 'Condition']
TARGET = 'Price_USD'

malaysia_aligned = malaysia_df[SHARED_FEATURES + [TARGET]].copy()
kaduna_aligned   = kaduna_df[SHARED_FEATURES + [TARGET]].copy()

print("Malaysia aligned shape:", malaysia_aligned.shape)
print("Kaduna aligned shape  :", kaduna_aligned.shape)

print("\nMalaysia Condition values:")
print(malaysia_aligned['Condition'].value_counts())

print("\nKaduna Condition values:")
print(kaduna_aligned['Condition'].value_counts())

Malaysia aligned shape: (37954, 6)
Kaduna aligned shape  : (37, 6)

Malaysia Condition values:
Condition
Fair    21788
New     11383
Old      4783
Name: count, dtype: int64

Kaduna Condition values:
Condition
New     19
Fair    11
Old      7
Name: count, dtype: int64


Confirm Feature Alignment

In [679]:
# Confirm feature alignment for both datasets
print("=" * 45)
print("       FEATURE ALIGNMENT CONFIRMATION")
print("=" * 45)

print(f"\nMalaysia aligned shape : {malaysia_aligned.shape}")
print(f"Kaduna aligned shape   : {kaduna_aligned.shape}")

print("\nMalaysia columns:")
print(malaysia_aligned.columns.tolist())

print("\nKaduna columns:")
print(kaduna_aligned.columns.tolist())

print("\nColumns match:", 
      malaysia_aligned.columns.tolist() == 
      kaduna_aligned.columns.tolist())

print("\nMalaysia sample:")
print(malaysia_aligned.head(3))

print("\nKaduna sample:")
print(kaduna_aligned.head(3))

print("\nMalaysia data types:")
print(malaysia_aligned.dtypes)

print("\nKaduna data types:")
print(kaduna_aligned.dtypes)

print("\nMissing values in Malaysia:")
print(malaysia_aligned.isnull().sum())

print("\nMissing values in Kaduna:")
print(kaduna_aligned.isnull().sum())

print("\n" + "=" * 45)
print("Feature alignment successfully confirmed.")
print("=" * 45)

       FEATURE ALIGNMENT CONFIRMATION

Malaysia aligned shape : (37954, 6)
Kaduna aligned shape   : (37, 6)

Malaysia columns:
['Bedrooms', 'Bathrooms', 'Size_sqm', 'Property_Type', 'Condition', 'Price_USD']

Kaduna columns:
['Bedrooms', 'Bathrooms', 'Size_sqm', 'Property_Type', 'Condition', 'Price_USD']

Columns match: True

Malaysia sample:
   Bedrooms  Bathrooms    Size_sqm Property_Type Condition  Price_USD
0       2.0        3.0  124.025505          Flat       New   312500.0
1       6.0        7.0  641.030700      Bungalow      Fair  1700000.0
2       3.0        4.0  174.193125          Flat      Fair   257500.0

Kaduna sample:
   Bedrooms  Bathrooms  Size_sqm Property_Type Condition     Price_USD
0         3          2       140      Bungalow       New  32000.000000
1         3          2       150      Bungalow       New  34666.666667
2         3          2       135      Bungalow      Fair  28666.666667

Malaysia data types:
Bedrooms         float64
Bathrooms        float64
Siz

Bedrooms         0
Bathrooms        0
Size_sqm         0
Property_Type    0
Condition        0
Price_USD        0
dtype: int64

Missing values in Kaduna:
Bedrooms         0
Bathrooms        0
Size_sqm         0
Property_Type    0
Condition        0
Price_USD        0
dtype: int64

Feature alignment successfully confirmed.


In [680]:
# Check what values are still in malaysia_aligned Condition
print("Malaysia aligned Condition unique values:")
print(malaysia_aligned['Condition'].unique())

print("\nKaduna aligned Condition unique values:")
print(kaduna_aligned['Condition'].unique())

Malaysia aligned Condition unique values:
<StringArray>
['New', 'Fair', 'Old']
Length: 3, dtype: str

Kaduna aligned Condition unique values:
<StringArray>
['New', 'Fair', 'Old']
Length: 3, dtype: str


In [681]:
from sklearn.preprocessing import LabelEncoder

for col in ['Property_Type', 'Condition']:
    le = LabelEncoder()

    combined = pd.concat([
        malaysia_aligned[col],
        kaduna_aligned[col]
    ]).astype(str)

    le.fit(combined)

    malaysia_aligned[col] = le.transform(
        malaysia_aligned[col].astype(str))
    kaduna_aligned[col]   = le.transform(
        kaduna_aligned[col].astype(str))

    print(f"{col} encoded — classes: {list(le.classes_)}")

print("\nMalaysia sample after encoding:")
print(malaysia_aligned.head(3))

print("\nKaduna sample after encoding:")
print(kaduna_aligned.head(3))

Property_Type encoded — classes: ['Bungalow', 'Duplex', 'Flat', 'Self-contain']
Condition encoded — classes: ['Fair', 'New', 'Old']

Malaysia sample after encoding:
   Bedrooms  Bathrooms    Size_sqm  Property_Type  Condition  Price_USD
0       2.0        3.0  124.025505              2          1   312500.0
1       6.0        7.0  641.030700              0          0  1700000.0
2       3.0        4.0  174.193125              2          0   257500.0

Kaduna sample after encoding:
   Bedrooms  Bathrooms  Size_sqm  Property_Type  Condition     Price_USD
0         3          2       140              0          1  32000.000000
1         3          2       150              0          1  34666.666667
2         3          2       135              0          0  28666.666667


 Prepare Feature Matrices

Prepare Feature Matrices" Means
It simply means separating your data into inputs and outputs so the model knows:

What to learn from (features/inputs)
What to predict (target/output)

 just organizing your data into inputs and outputs before feeding it to the model.

In [682]:
# Malaysia
X_malaysia = malaysia_aligned[SHARED_FEATURES]
y_malaysia  = malaysia_aligned[TARGET]

# Kaduna
X_kaduna = kaduna_aligned[SHARED_FEATURES]
y_kaduna  = kaduna_aligned[TARGET]

print("X_malaysia shape:", X_malaysia.shape)
print("y_malaysia shape:", y_malaysia.shape)

print("\nX_kaduna shape:", X_kaduna.shape)
print("y_kaduna shape:", y_kaduna.shape)

print("\nMalaysia features sample:")
print(X_malaysia.head(3))

print("\nKaduna features sample:")
print(X_kaduna.head(3))

print("\nReady for transfer learning.")

X_malaysia shape: (37954, 5)
y_malaysia shape: (37954,)

X_kaduna shape: (37, 5)
y_kaduna shape: (37,)

Malaysia features sample:
   Bedrooms  Bathrooms    Size_sqm  Property_Type  Condition
0       2.0        3.0  124.025505              2          1
1       6.0        7.0  641.030700              0          0
2       3.0        4.0  174.193125              2          0

Kaduna features sample:
   Bedrooms  Bathrooms  Size_sqm  Property_Type  Condition
0         3          2       140              0          1
1         3          2       150              0          1
2         3          2       135              0          0

Ready for transfer learning.


Train GBR on Malaysia (Source Domain)

In [683]:
from sklearn.ensemble import GradientBoostingRegressor

gbr_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)

gbr_model.fit(X_malaysia, y_malaysia)
print("GBR model trained successfully on Malaysia dataset.")

print("\nFeature importances:")
for feat, imp in zip(SHARED_FEATURES,
                     gbr_model.feature_importances_):
    print(f"  {feat:20s}: {imp:.4f}")

GBR model trained successfully on Malaysia dataset.

Feature importances:
  Bedrooms            : 0.0630
  Bathrooms           : 0.4541
  Size_sqm            : 0.1649
  Property_Type       : 0.2534
  Condition           : 0.0646
